In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- rft_concat_write ---
RFT_CONCAT_BASE_PD = pd.DataFrame({"Realization":[1,2],"Well":["W1","W1"],"Ensemble":["E1","E1"],"Iteration":[1,1],"value":[5.0,6.0]})
RFT_CONCAT_BASE_PL = pl.from_pandas(RFT_CONCAT_BASE_PD)
FIX_RFT_CONCAT_WRITE_DROP_CONST_COLS = False
FIX_RFT_CONCAT_WRITE_OUTPUT_FILE = "test_file.csv"

def make_rft_concat_write_data_pd():
    return [RFT_CONCAT_BASE_PD.copy()]

def make_rft_concat_write_data_pl():
    return [RFT_CONCAT_BASE_PL.clone()]

# --- rft_obs_join ---
FIX_RFT_OBS_JOIN_OBS_NODE = pd.DataFrame({"observations": [10.0], "std": [0.5]})
FIX_RFT_OBS_JOIN_PRESSURE_VALS = np.array([1.0, 2.0, 3.0])
FIX_RFT_OBS_JOIN_REALIZATIONS = [0]
FIX_RFT_OBS_JOIN_TVD_ARG = [100., 200., 300.]
FIX_RFT_OBS_JOIN_WELL = "W1"

def make_rft_obs_join_data_pd():
    return []

def make_rft_obs_join_data_pl():
    return []

def make_rft_obs_join_realization_frame_pd():
    return pd.DataFrame({"date":["2020-01-01"],"value":[100.0]})

def make_rft_obs_join_realization_frame_pl():
    return pl.DataFrame({"date":["2020-01-01"],"value":[100.0]})

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_rft_concat_write(data, drop_const_cols, output_file):
    frame = pd.concat(data)
    frame.set_index(["Realization", "Well", "Ensemble", "Iteration"], inplace=True)
    if drop_const_cols:
        frame = frame.loc[:, (frame != frame.iloc[0]).any()]
    frame.to_csv(output_file)
    return frame

def before_rft_obs_join(data, obs_node, pressure_vals, realizations, tvd_arg, well, realization_frame):
    rft_data = pd.DataFrame(pressure_vals, index=range(len(tvd_arg)))
    ensemble_data = []
    for iens in realizations:
        frame = pd.DataFrame(
            data={"TVD": tvd_arg, "Pressure": rft_data[iens],
                  "ObsValue": obs_node["observations"].values[0],
                  "ObsStd": obs_node["std"].values[0]},
        )
        realization_frame["Realization"] = iens
        realization_frame["Well"] = well
        ensemble_data.append(realization_frame)
    data.append(pd.concat(ensemble_data))
    return ensemble_data

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_rft_concat_write(data, drop_const_cols, output_file):
    frame = pl.concat(data)
    if drop_const_cols:
        frame = frame.select([col for col in frame.columns if (frame[col] != frame[col][0]).any()])
    frame.write_csv(output_file)
    return frame

def gen_rft_obs_join(data, obs_node, pressure_vals, realizations, tvd_arg, well, realization_frame):
    rft_data = pl.DataFrame(pressure_vals)
    ensemble_data = []
    for iens in realizations:
        realization_frame = pl.DataFrame(
            data={"TVD": tvd_arg, "Pressure": rft_data[iens],
                  "ObsValue": obs_node["observations"].values[0],
                  "ObsStd": obs_node["std"].values[0]},
        )
        realization_frame = realization_frame.with_columns(
            pl.lit(iens).alias("Realization"),
            pl.lit(well).alias("Well"),
        )
        ensemble_data.append(realization_frame)
    data.append(pl.concat(ensemble_data))
    return ensemble_data

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def _comparison_label(label):
    text = str(label)
    if text.lstrip().startswith(("L2", "L3")):
        return text
    return f"L2 equivalence {text}"

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:

# === Tests: rft_concat_write ===
import tempfile

def _run_rft_write(func, data, drop_const_cols):
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "rft.csv"
        result = func(data, drop_const_cols, str(path))
        text = path.read_text(encoding="utf-8") if path.exists() else None
        return result, text

try:
    _r, _ = _run_rft_write(gen_rft_concat_write, make_rft_concat_write_data_pl(), False)
    print("✅ L1 smoke gen_rft_concat_write: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_rft_concat_write: {type(_e).__name__}: {_e}")

try:
    _rb, _ = _run_rft_write(before_rft_concat_write, make_rft_concat_write_data_pd(), False)
    print("✅ L1 smoke before_rft_concat_write: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_rft_concat_write: {type(_e).__name__}: {_e}")

for _layer, _drop in [("L2 equivalence", False), ("L3 edge", True)]:
    try:
        _rb, _before_text = _run_rft_write(before_rft_concat_write, make_rft_concat_write_data_pd(), _drop)
        _rg, _gen_text = _run_rft_write(gen_rft_concat_write, make_rft_concat_write_data_pl(), _drop)
        compare(_rb.reset_index(), _rg, f"{_layer} rft_concat_write frame", check_row_order=True)
        if _before_text == _gen_text:
            print(f"✅ {_layer} rft_concat_write CSV contents: MATCH")
        else:
            print(f"❌ {_layer} rft_concat_write CSV contents: MISMATCH — before={_before_text!r}, gen={_gen_text!r}")
    except Exception as _e:
        print(f"❌ {_layer} rft_concat_write: {type(_e).__name__}: {_e}")

try:
    _before_error = _gen_error = None
    try: _run_rft_write(before_rft_concat_write, [], False)
    except Exception as _e: _before_error = _e
    try: _run_rft_write(gen_rft_concat_write, [], False)
    except Exception as _e: _gen_error = _e
    if _before_error is not None and _gen_error is not None and not isinstance(_gen_error, (SyntaxError, NameError)):
        print(f"✅ L3 edge rft_concat_write empty list: both sides reject (before={type(_before_error).__name__}, gen={type(_gen_error).__name__})")
    else:
        print(f"❌ L3 edge rft_concat_write empty list: MISMATCH — before={_before_error}, gen={_gen_error}")
except Exception as _e:
    print(f"❌ L3 edge rft_concat_write empty list: {type(_e).__name__}: {_e}")
